In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict
import pybedtools

In [2]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
HIMORNA_INITIAL = DATA_INITIAL / "himorna"
HIMORNA = DATA / "himorna"
HM_INITIAL = HIMORNA_INITIAL / "histone_modifications"
HM = HIMORNA / "histone_modifications"
METHYL_INITIAL = HIMORNA_INITIAL / "methylation"
METHYL = HIMORNA / "methylation"
FANTOM = DATA / "fantom_cat"
PREPROCESSING = Path("../preprocessing")
HIMORNA_INDEX = PREPROCESSING / "himorna_index"
HIMORNA_GENES_SIGN =  PREPROCESSING / "himorna_genes_sign"
HIMORNA_INDEX_BED = PREPROCESSING / "himorna_index-bed"

for directory in DATA, HIMORNA, HIMORNA_NEW:
    directory.mkdir(exist_ok=True, parents=True)

to_himorna_peaks = HM_INITIAL / "Himorna_all.tsv.gz"
himorna_peaks = pd.read_csv(
        to_himorna_peaks,
        chunksize=100000000,
        sep="\t",
    )
himorna_peaks = pd.concat(himorna_peaks)
print("himorna_peaks: done")

himorna_peaks

lnc_rna_unique = himorna_peaks["lnc"].unique()
len(lnc_rna_unique)

lnc_rna_unique

lnc_rna_ctd = [i for i in lnc_rna_unique if "CTD-" in i]

ENSG00000267577 = himorna_peaks[himorna_peaks["lnc_ens"] == "ENSG00000267577"]

ENSG00000267577.to_csv("ENSG00000267577.tsv.gz", sep="\t", compression="gzip")

ENSG00000267577["hm"].value_counts()

anna_budhimorna_peaks[himorna_peaks["lnc"] == "CTD-2587H24"]

mask = himorna_peaks["lnc"].apply(lambda x: "CTD-2587H24" in x)

himorna_peaks[mask]

In [3]:
to_himorna_peaks = HIMORNA / "himorna_peaks-hm_meth-gencode47-assoc-l2permissive_hg38.tsv.gz"
himorna_peaks_assoc = pd.read_csv(
        to_himorna_peaks,
        chunksize=100000000,
        sep="\t",
    )
himorna_peaks_assoc = pd.concat(himorna_peaks_assoc)
print("himorna_peaks_assoc: done")

KeyboardInterrupt: 

In [ ]:
himorna_peaks_assoc.head()

In [5]:
himorna_peaks_assoc.shape

(705240949, 10)

In [7]:
himorna_peaks_assoc = himorna_peaks_assoc.dropna(subset=["hm","lnc_ens"])
himorna_peaks_assoc.shape

(705240949, 10)

In [13]:
himorna_peaks_assoc_bed = himorna_peaks_assoc[["hm", "lnc_ens", "chrom", "start", "end"]] 

In [20]:
def indexing_himorna_by_bed(hm, lnc_ens, subdf):
    to_index = HIMORNA_INDEX_BED / hm
    to_index.mkdir(exist_ok=True, parents=True)
    to_save = to_index / f"{hm}-{lnc_ens}-sort.bed"
    if to_save.exists():
        print(f"Skipped: {hm} {lnc_ens}")
        return None
    subdf_bed = pybedtools.BedTool.from_dataframe(subdf)
    subdf_bed_sortbed = subdf_bed.sort()
    subdf_bed_sortbed.moveto(to_save)

In [21]:
with Parallel(n_jobs=-1) as pool:
    chunks = pool(
        delayed(indexing_himorna_by_bed)(hm, lnc_ens, subdf[["chrom", "start", "end"]])
        for (hm, lnc_ens), subdf in himorna_peaks_assoc_bed.groupby(by=["hm", "lnc_ens"])
    )

In [15]:
himorna_peaks_assoc["corr_sign"] = np.where(himorna_peaks_assoc["corr"] > 0, "plus", "minus")

In [22]:
himorna_peaks_assoc = himorna_peaks_assoc.dropna(subset=["hm","lnc_ens","gene_ens"])
himorna_peaks_assoc.shape

(566743909, 11)

In [24]:
himorna_peaks_assoc[["hm", "lnc_ens", "chrom", "gene_ens", "gene_chrom"]] = himorna_peaks_assoc[["hm", "lnc_ens", "chrom", "gene_ens", "gene_chrom"]].astype('str')

/tmp/ipykernel_1891849/779899520.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  himorna_peaks_assoc[["hm", "lnc_ens", "chrom", "gene_ens"]] = himorna_peaks_assoc[["hm", "lnc_ens", "chrom", "gene_ens"]].astype('str')


In [26]:
himorna_peaks_assoc[["start", "end", "gene_start", "gene_end"]] = himorna_peaks_assoc[["start", "end", "gene_start", "gene_end"]].astype(np.uint32)

/tmp/ipykernel_1891849/1031985711.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  himorna_peaks_assoc[["start", "end", "gene_start", "gene_end"]] = himorna_peaks_assoc[["start", "end", "gene_start", "gene_end"]].astype(np.uint32)


In [46]:
himorna_peaks_assoc["gene_ens"] = himorna_peaks_assoc["gene_ens"].apply(lambda x: x.split("-")[0])

In [48]:
himorna_peaks_assoc

,hm,lnc_ens,chrom,start,end,corr,gene_chrom,gene_start,gene_end,gene_ens,corr_sign
36,H3K27ac,ENSG00000280018,chr18,10446693,10447377,0.579,chr18,10448312,10476078,CATG00000036918,plus
37,H3K27ac,ENSG00000269416,chr18,10446693,10447377,0.411,chr18,10448312,10476078,CATG00000036918,plus
38,H3K27ac,ENSG00000242797,chr18,10446693,10447377,0.420,chr18,10448312,10476078,CATG00000036918,plus
39,H3K27ac,ENSG00000233056,chr18,10446693,10447377,0.530,chr18,10448312,10476078,CATG00000036918,plus
40,H3K27ac,ENSG00000279148,chr18,10448004,10448179,-0.525,chr18,10448312,10476078,CATG00000036918,minus
...,...,...,...,...,...,...,...,...,...,...,...
705240944,methylation,ENSG00000261222,chr8,9741473,9741474,-0.890,chr8,9551766,9788424,ENSG00000173273,minus
705240945,methylation,ENSG00000261222,chr8,9741473,9741474,-0.890,chr8,9740848,9741768,ENSG00000207701,minus
705240946,methylation,ENSG00000261222,chr8,10827627,10827628,-0.879,chr8,10712751,10839868,ENSG00000171056,minus
705240947,methylation,ENSG00000261222,chr8,10827627,10827628,-0.879,chr8,10725399,10839868,ENSG00000258724,minus


In [49]:
def indexing_himorna(hm, lnc_ens, corr_sign, subdf):
    to_index = HIMORNA_INDEX / hm
    to_index.mkdir(exist_ok=True, parents=True)
    to_save = to_index / f"{hm}-{lnc_ens}-{corr_sign}-.tsv.gz"
    if to_save.exists():
        print(f"Skipped: {hm} {lnc_ens} {corr_sign}")
        return None
    subdf.to_csv(to_save, sep="\t", compression="gzip", index=False)

In [50]:
with Parallel(n_jobs=-1) as pool:
    chunks = pool(
        delayed(indexing_himorna)(hm, lnc_ens, corr_sign, subdf)
        for (hm, lnc_ens, corr_sign), subdf in himorna_peaks_assoc.groupby(by=["hm", "lnc_ens", "corr_sign"])
    )

In [9]:
H3K36me3_ENSG00000267110 = pd.concat([pd.read_csv(to_file, sep="\t") for to_file in HIMORNA_INDEX.glob("H3K36me3/*ENSG00000267110*")])
H3K36me3_ENSG00000267110

ValueError: No objects to concatenate

In [10]:
ENSG00000267110 = pd.concat([pd.read_csv(to_file, sep="\t") for to_file in HIMORNA_INDEX.glob("*/*ENSG00000267110*")])
ENSG00000267110

ValueError: No objects to concatenate

In [51]:
def indexing_himorna_genes(hm, lnc_ens):
    """
    Это верный способ извлекать plus и minus скоррелирвоанные гены. Посколько наивный подход не учитывает то, 
    что один и тот же ген может иметь как plus, так и minus пики. Данная функция приписывает знак гену по принципу "Тот знак, где больше пиков"
    """
    df = pd.concat([pd.read_csv(i, sep="\t") for i in HIMORNA_INDEX.glob(f'{hm}/{hm}-{lnc_ens}-*-.tsv.gz')])
    genes_ens_dict = defaultdict(list)
    for gene_ens, subdf in df.groupby(by=["gene_ens"]):
        corr_sign = subdf.value_counts(subset=["corr_sign"]).idxmax()[0]
        genes_ens_dict[corr_sign].append(gene_ens[0])
    
    for corr_sign in genes_ens_dict:
        print(corr_sign)
        to_index = HIMORNA_GENES_SIGN / hm
        to_index.mkdir(exist_ok=True, parents=True)
        to_save = to_index / f"{hm}-{lnc_ens}-{corr_sign}-genes_list-.tsv.gz"
        pd.DataFrame({"genes_list":genes_ens_dict[corr_sign]}).to_csv(to_save, sep="\t", compression="gzip", index=False)

In [52]:
list_of_hm = himorna_peaks_assoc["hm"].unique()

In [55]:
list_of_hm

array(['H3K27ac', 'H3K27me3', 'H3K36me3', 'H3K4me1', 'H3K4me2', 'H3K4me3',
       'H3K79me2', 'H3K9ac', 'H3K9me3', 'H4K20me1', 'methylation'],
      dtype=object)

In [53]:
with Parallel(n_jobs=-1) as pool:
    chunks = pool(
        delayed(indexing_himorna_genes)(hm, lnc_ens)
        for hm in list_of_hm for lnc_ens in [i.stem.split("-")[1] for i in HIMORNA_INDEX.glob(f'{hm}/{hm}-*-.tsv.gz')]
    )

In [54]:
to_himorna = HIMORNA_GENES_SIGN / "H3K27ac/H3K27ac-ENSG00000223813-plus-genes_list-.tsv.gz"
plus_genes_H3K27ac_ENSG00000223813 = pd.read_csv(to_himorna, sep="\t")
plus_genes_H3K27ac_ENSG00000223813

,genes_list
0,CATG00000000117
1,CATG00000000173
2,CATG00000000331
3,CATG00000000490
4,CATG00000000496
...,...
2878,ENSG00000272848
2879,ENSG00000272865
2880,ENSG00000272917
2881,ENSG00000273274


plus
minus
minus
plus
minus
plus
plus
minus
plus
plus
minus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
plus
minus
plus
minus
plus
plus
plus
minus
plus
plus
minus
minus
plus
minus
plus
plus
plus
plus
plus
minus
minus
plus
minus
plus
minus
plus
plus
minus
plus
minus
minus
minus
plus
minus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
minus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
minus
plus
minus
plus
minus
minus
plus
minus
plus
minus
plus
plus
minus
minus
minus
plus
plus
minus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
minus
plus
plus
minus
plus
minus
minus
plus
plus
minus
minus
plus
minus
plus
plus
plus
minus
plus
minus
minus
plus
plus
minus
plus
minus
plus
minus
minus
plus


In [ ]:
# проверка общего количество генов

In [36]:
to_file = FANTOM / 'fantom_l2-hg19-sort.bed'
fantom_hg19 = pd.read_csv(to_file, sep="\t",  names=["chrom","start","end", "name"], header=0)
mask = fantom_hg19["name"].apply(lambda x: "gene" in x)
fantom_genes_hg19 = fantom_hg19[mask]

to_file = FANTOM / 'fantom_l2-hg38-sort.bed'
fantom_hg38 = pd.read_csv(to_file, sep="\t",  names=["chrom","start","end", "name"], header=0)
mask = fantom_hg38["name"].apply(lambda x: "gene" in x)
fantom_genes_hg38 = fantom_hg38[mask]

In [39]:
himorna_peaks_assoc_genes = himorna_peaks_assoc["gene_ens"].unique()
fantom_genes_hg19_genes = fantom_genes_hg19["name"].unique()
fantom_genes_hg38_genes = fantom_genes_hg38["name"].unique()

In [44]:
len(himorna_peaks_assoc_genes), len(fantom_genes_hg19_genes), len(fantom_genes_hg38_genes)

(123893, 124245, 123997)